In [ ]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 49.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from rdkit import Chem

# Загрузка фильтров Glaxo
glaxo_filters = pd.read_csv('glaxo_filters.csv', sep=';')
# Создаем объекты Mol для каждого SMARTS фильтра
glaxo_patterns = [(row['smarts'], Chem.MolFromSmarts(row['smarts'])) for _, row in glaxo_filters.iterrows()]

# Загрузка файла с молекулами (замените 'molecules.csv' на ваше название файла)
# Предполагается, что в файле есть колонка 'SMILES'
df = pd.read_csv('molecules.csv')

In [ ]:
def calc_glaxo(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False, ""

    found_structs = []
    for smarts_str, pattern in glaxo_patterns:
        if pattern and mol.HasSubstructMatch(pattern):
            found_structs.append(smarts_str)

    if found_structs:
        return True, found_structs
    return False, found_structs

In [ ]:
def apply_filters(row):
    smiles = row['SMILES']
    glaxo_res = calc_glaxo(smiles)

    # Выпаковываем результаты (bool, list)
    row['Glaxo'] = glaxo_res[0] if glaxo_res else None
    row['Glaxo_structs'] = ", ".join(glaxo_res[1]) if glaxo_res and glaxo_res[1] else ""

    return row

In [ ]:
df = df.apply(apply_filters, axis=1)

# Сохраняем результат
df.to_csv('molecules_with_glaxo_filters.csv', index=False)
print("Обработка завершена. Результаты сохранены в molecules_with_glaxo_filters.csv")
df.head()

Обработка завершена. Результаты сохранены в molecules_with_glaxo_filters.csv


,Name A,SMILES,Structural difference,Comparison type,More stable molecule,stability_mechanism,reason,IUPAC,QED,SA,MW,LogP,Lipinski,BRENK,PAINS,Glaxo,BRENK STRUCTS,PAINS STRUCTS,Glaxo_structs
0,METHYL ACETATE,CC(=O)OC,ester vs amide,hydrolytic_stability,less stable bro,hydrolysis,Амид стабилизирован за счет резонанса и менее ...,methyl acetate,"0,38","1,74","74,08","0,18",True,False,False,False,NaN,NaN,
1,N-METHYL ACETAMIDE,CC(=O)NC,NaN,hydrolytic_stability,more stable bro,NaN,NaN,N-methylacetamide,"0,42","1,91","73,09","-0,25",True,False,False,False,NaN,NaN,
2,ETHYL BENZOATE,CCOC(=O)C1=CC=CC=C1,ester vs amide,hydrolytic_stability,less stable bro,hydrolysis,"Амид лучше противостоит нуклеофильной атаке, ч...",ethyl benzoate,"0,6","1,14","150,18","1,86",True,False,False,False,NaN,NaN,
3,BENZAMIDE,C1=CC=C(C=C1)C(=O)N,NaN,hydrolytic_stability,more stable bro,NaN,NaN,benzamide,"0,59","1,16","121,14","0,79",True,False,False,False,NaN,NaN,
4,PHENYL ACETATE,CC(=O)OC1=CC=CC=C1,ester vs amide,hydrolytic_stability,less stable bro,hydrolysis,Амидная связь менее подвержена гидролизу,phenyl acetate,"0,43","1,31","136,15","1,61",True,True,False,False,phenol_ester,NaN,


## Ручная проверка

In [ ]:
smiles = "O=C(NC(CO)C(O)c1ccc([N+](=O)[O-])cc1)C(Cl)Cl"
print(calc_glaxo(smiles))

(True, ['[Br,Cl,I][CX4;CH,CH2]'])
